In [1]:
# %load_ext autoreload
# %autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add parent directory to path so we can import the 'deep_ecg' modules
sys.path.append(os.path.abspath(os.path.join('..')))

from load_dataset import load_nsrdb_dataset
from run import (
    run_closed_set_identification,
    run_verification,
    run_subject_disjoint_verification,
    run_cross_session_identification,
    run_cross_session_verification
)
from models import DeepECG, ResNet1D 
from visualizations import Visualizer 

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

Using device: cpu


In [2]:
print("\n=== TASKS 1-3: Random Split (Baseline) ===")
print("Loading NSRDB (First Hour Only) for Random Split to save RAM...")

# 1 Hour is approx 0.042 of the file (1/24)
loader = load_nsrdb_dataset(
    num_beats=3, 
    enrollment_range=(0.0, 0.042), # Load only 1st hour
    cleanup_zip=False
)
# We use load_session("Session_1") to just get that specific hour
x_all, y_all = loader.load_session("Session_1") 

print(f"Data Loaded: {x_all.shape} samples, {len(np.unique(y_all))} subjects")

# Run Standard Benchmarks
print("\n[Task 1] Closed-Set Identification")
run_closed_set_identification(x_all, y_all, DeepECG, epochs=2, device=device)

print("\n[Task 2] Verification")
run_verification(x_all, y_all, DeepECG, epochs=2, device=device)

print("\n[Task 3] Subject-Disjoint Verification")
run_subject_disjoint_verification(x_all, y_all, DeepECG, epochs=2, device=device)


=== TASKS 1-3: Random Split (Baseline) ===
Loading NSRDB (First Hour Only) for Random Split to save RAM...


Processing Session_1: 100%|██████████| 18/18 [09:29<00:00, 31.67s/it]


Data Loaded: (91510, 77) samples, 18 subjects

[Task 1] Closed-Set Identification

[TASK] Closed-Set Identification on cpu
    Epoch 001 | Loss: 0.3794
    Epoch 002 | Loss: 0.0682
[RESULT] Closed-Set Accuracy: 0.9864

[Task 2] Verification

[TASK] Verification (Random Split / Closed-Set) on cpu
[INFO] Phase 1: Training feature extractor (Learning Identity)...
    Epoch 001 | Loss: 0.4210
    Epoch 002 | Loss: 0.0775
[INFO] Phase 2: Computing EER using 'balanced' sampling...
[INFO] Generating 10000 BALANCED pairs (50/50 split)...
[RESULT] Mode: BALANCED | EER: 0.0364 | AUC: 0.9936

[Task 3] Subject-Disjoint Verification

[TASK] Subject-Disjoint Verification (Open-Set) on cpu
[INFO] Splitting: 12 Training Subjects vs 6 Test Subjects
[INFO] Phase 1: Training on Known Subjects...
    Epoch 001 | Loss: 0.2797
    Epoch 002 | Loss: 0.0428
[INFO] Phase 2: Computing EER on 6 Unseen Subjects...
[INFO] Generating 10000 BALANCED pairs (50/50 split)...
[RESULT] Mode: BALANCED | EER: 0.1484 | AUC:

{'eer': 0.1484000000004833, 'auc': 0.9348225200000001}

In [3]:
print("\n=== TASK 4: Biometric Regimes ===")

# Ratio for 1 Hour: 1/24 ≈ 0.0417

# A. SHORT-TERM (Consecutive Hours)
# Enroll: Hour 0-1
# Probe:  Hour 1-2
print("\n[A] Loading Short-Term Regime (Hour 0-1 vs Hour 1-2)...")
loader_st = load_nsrdb_dataset(
    num_beats=3, 
    enrollment_range=(0.0, 0.042),   # Hour 1
    probe_range=(0.042, 0.084),      # Hour 2
    cleanup_zip=False
)
x_st_enr, y_st_enr = loader_st.load_session("Session_1")
x_st_prb, y_st_prb = loader_st.load_session("Session_2")
print(f"    Short-Term Shapes: Enroll {x_st_enr.shape}, Probe {x_st_prb.shape}")

print("    Running Verification...")
run_cross_session_verification(x_st_enr, y_st_enr, x_st_prb, y_st_prb, DeepECG, epochs=2, device=device, visualize=False)


# B. LONG-TERM (Circadian / First vs Last)
# Enroll: Hour 0-1
# Probe:  Hour 23-24 (Last Hour)
print("\n[B] Loading Circadian Regime (Hour 0-1 vs Hour 23-24)...")
loader_lt = load_nsrdb_dataset(
    num_beats=3, 
    enrollment_range=(0.0, 0.042),   # First Hour
    probe_range=(0.958, 1.0),        # Last Hour
    cleanup_zip=False
)
x_lt_enr, y_lt_enr = loader_lt.load_session("Session_1")
x_lt_prb, y_lt_prb = loader_lt.load_session("Session_2")
print(f"    Long-Term Shapes: Enroll {x_lt_enr.shape}, Probe {x_lt_prb.shape}")

print("    Running Verification...")
run_cross_session_verification(x_lt_enr, y_lt_enr, x_lt_prb, y_lt_prb, DeepECG, epochs=2, device=device, visualize=False)


=== TASK 4: Biometric Regimes ===

[A] Loading Short-Term Regime (Hour 0-1 vs Hour 1-2)...


Processing Session_2: 100%|██████████| 18/18 [07:43<00:00, 25.78s/it]


    Short-Term Shapes: Enroll (91510, 77), Probe (89564, 77)
    Running Verification...

[TASK] Cross-Session Verification (EER) on cpu
[INFO] Phase 1: Training Feature Extractor on Session 1...
    Epoch 001 | Loss: 0.3278
    Epoch 002 | Loss: 0.0621
[INFO] Phase 2: Extracting Embeddings for Pairing...
[INFO] Generating 10000 Cross-Session Pairs (balanced)...
[RESULT] Cross-Session EER: 0.0312 | AUC: 0.9950

[B] Loading Circadian Regime (Hour 0-1 vs Hour 23-24)...


Processing Session_2: 100%|██████████| 18/18 [11:19<00:00, 37.72s/it]


    Long-Term Shapes: Enroll (91510, 77), Probe (132958, 77)
    Running Verification...

[TASK] Cross-Session Verification (EER) on cpu
[INFO] Phase 1: Training Feature Extractor on Session 1...
    Epoch 001 | Loss: 0.3278
    Epoch 002 | Loss: 0.0621
[INFO] Phase 2: Extracting Embeddings for Pairing...
[INFO] Generating 10000 Cross-Session Pairs (balanced)...
[RESULT] Cross-Session EER: 0.4962 | AUC: 0.5048


{'eer': 0.4962, 'auc': 0.5048364}

In [4]:
print("\n=== TASK 5: Blind Segmentation (Circadian Regime) ===")

blind_params = {
    'mode': 'blind',
    'window_len': 5.0,  
    'stride': 2.0,      
    'bandpass': True,
    'normalize': 'zscore'
}

# Use Circadian Split (First vs Last Hour) to test blindness + aging
loader_blind = load_nsrdb_dataset(
    num_beats=1, 
    enrollment_range=(0.0, 0.042),
    probe_range=(0.958, 1.0),
    preprocessing_params=blind_params, 
    cleanup_zip=False
)

x_blind_enr, y_blind_enr = loader_blind.load_session("Session_1")
x_blind_prb, y_blind_prb = loader_blind.load_session("Session_2")
print(f"Blind Data Shapes: Enroll {x_blind_enr.shape}, Probe {x_blind_prb.shape}")

print("\n[Blind] Identification")
run_cross_session_identification(x_blind_enr, y_blind_enr, x_blind_prb, y_blind_prb, DeepECG, epochs=2, device=device)

print("\n[Blind] Verification")
run_cross_session_verification(x_blind_enr, y_blind_enr, x_blind_prb, y_blind_prb, DeepECG, epochs=2, device=device, visualize=False)


=== TASK 5: Blind Segmentation (Circadian Regime) ===


Processing Session_2: 100%|██████████| 18/18 [00:01<00:00, 17.74it/s]


Blind Data Shapes: Enroll (33037, 640), Probe (33037, 640)

[Blind] Identification

[TASK] Cross-Session Identification (Rank-1) on cpu
[INFO] Subjects: 18 in Train, 18 in Test.
[INFO] Evaluated on intersection: 18 common subjects.
[INFO] Phase 1: Training Classifier on Session 1...
    Epoch 001 | Loss: 0.5916
    Epoch 002 | Loss: 0.0336
[INFO] Phase 2: Predicting on Session 2...
[RESULT] Cross-Session Identification Accuracy: 5.07%

[Blind] Verification

[TASK] Cross-Session Verification (EER) on cpu
[INFO] Phase 1: Training Feature Extractor on Session 1...
    Epoch 001 | Loss: 0.5916
    Epoch 002 | Loss: 0.0336
[INFO] Phase 2: Extracting Embeddings for Pairing...
[INFO] Generating 10000 Cross-Session Pairs (balanced)...
[RESULT] Cross-Session EER: 0.4710 | AUC: 0.5373


{'eer': 0.47100000000000025, 'auc': 0.53732858}